# ME 323 Beam-Design Activity — Instructor Walkthrough
## From three failure equations to a design decision under uncertainty

This walks the curriculum team through the full Module-1 arc: the mechanics students use, how well it lines up with real test data, where it falls apart, what the test stand does to it, and why that gap is what the Gaussian-Process work in the companion notebook is for.

### The idea
We want students to see that ML tools earn their keep when the physics is messy and the data is thin. The beam problem fits. The most common failure on these printed parts is brittle fracture along the print layers, and no equation predicts it. Lateral-torsional buckling depends as much on the test stand as on the beam. The failure modes overlap. So the equations cannot hand you the answer, which is the setup we want.

### The arc students follow
1. Meet the three failure equations (bending, shear, LTB) next to the textbook "vanilla" versions (sections 3–4, compared in 6.5).
2. Check the theory against the test data: which mode drives, does it match, do the modes blur (sections 6–7.5).
3. Watch the same beam give two LTB answers on two stands (section 5).
4. Compare the failures they saw to the theory and adjust the constants, like the LTB length factor `k`, to fit (section 8.5).
5. Pull a mass-optimized design out of the theory, and watch the vanilla LTB calc point at the wrong beam (section 9).
6. Pick a design from a plain Gaussian Process in the companion notebook `ME323_Module1_Virtual_Lab_A` (section 10).
7. Move to a physics-informed GP, weigh explore against exploit, and choose between two acquisition rules (MUI vs. EI).
8. Write a memo defending the choice. The memo is what gets graded.

### What it teaches, and what it leans on
The point is engineering judgment under uncertainty, not ML theory. GP and BO show up as tools. Students already have the pieces from two courses, so we point back rather than reteach.
- **ME 239** has the uncertainty side: Bayes' rule, Gaussians and multivariate Gaussians, conditioning a Gaussian on data, uncertainty quantification. A GP is conditioning a multivariate Gaussian on data, scaled up to a whole function.
- **ME 323** has the mechanics: shear and bending-moment diagrams, flexural stress $\sigma=My/I$, what $I$ means, why I-beams, shear stress.
- **New here, kept light:** the Gaussian Process, Bayesian optimization, optimal experimental design, plus lateral-torsional buckling and the failure-mode picture below.

The notebook is a guided activity. The memo is the graded artifact.

We use two datasets: an original set (16 mm flange, 25 mm tall, older stand) and a new set (12 mm × 18 mm, Purdue Instron). The running thread is simple: these closed-form calcs are useful but partial, and seeing where they miss is what sends us to a data-driven surrogate.

## 1. Assumptions and constants

Every number below is an assumption. Change them and the predictions change. They sit here so nothing hides inside a function.

**Material (3-D-printed PLA, nominal):**
- Young's modulus $E = 2.5\;\text{GPa}$
- Poisson's ratio $\nu = 0.35$
- Shear modulus $G = E/2.6$  (close to $E/[2(1+\nu)]$)
- Tensile yield $\sigma_y = 76\;\text{MPa}$ — the **normal defined yield**, used for both bending and von Mises shear

**Lateral-torsional buckling factors** (simply-supported beam, single central point load):
- Moment-gradient factor $C_1 = 1.35$
- Load-height factor $C_2 = 0.55$
- Effective-length factor $k = L_b/L$ — **this is fixture-dependent**, see section 5

**Geometry conventions** (doubly-symmetric I-section):
- $b$ = web thickness, $H_\text{web}$ = clear web height between flanges
- $B$ = flange width (fixed per dataset), $TH$ = **total** section height (fixed per dataset)
- flange thickness $t_f = (TH - H_\text{web})/2$, extreme-fiber distance $c = TH/2$
- $L$ = test span

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

# ---------- MATERIAL (assumed) ----------
E_MODULUS = 2.5e9      # Pa, Young's modulus
POISSON   = 0.35       # -, Poisson's ratio
G_MODULUS = E_MODULUS / 2.6   # Pa, shear modulus (G = E/2.6)
YIELD_STR = 76e6       # Pa, NORMAL defined tensile yield

# ---------- LTB factors (central point load) ----------
C1_LTB = 1.35          # moment-gradient factor
C2_LTB = 0.55          # load-height factor

# ---------- Effective-length factor k = L_b / L  (FIXTURE-DEPENDENT) ----------
K_PURDUE   = 0.33      # CALIBRATED to the Purdue/Instron stand (fit to the tipping beams)
K_ORIGINAL = 0.56      # CALIBRATED to the 7 recorded "tipped" original beams (warping + top-flange
                       # load), ~5% error on their LOADS. k_orig=0.56 > Purdue 0.33 -> old stand LESS
                       # restrained ("less stable"). NOTE: applied box-wide this single k predicts more
                       # LTB-governed beams than actually tipped -- many old beams sit on the bend<->LTB
                       # boundary, so the split is very k-sensitive (12 LTB at 0.5, ~21 at 0.56). A KNOB.

# ---------- Per-dataset fixed geometry + fixture ----------
#   B  = flange width (mm), TH = total height (mm), L = span (mm), k = effective-length factor
BOXES = {
    "Original (16x25, old stand)": dict(B=16.0, TH=25.0, L=203.2, k=K_ORIGINAL, marker="^"),
    "Purdue (12x18, Instron)":     dict(B=12.0, TH=18.0, L=200.0, k=K_PURDUE,   marker="o"),
}
print("Constants loaded. sigma_y =", YIELD_STR/1e6, "MPa,  E =", E_MODULUS/1e9, "GPa,  G =", round(G_MODULUS/1e9,3), "GPa")


## 2. Load both datasets from GitHub

Both sets load directly from GitHub raw links. A local fallback is included for the Purdue set in case the link is unreachable.

Both reduce to the same columns: web thickness `b`, web height `H_web`, and the measured peak force `P_meas` (the failure load). The original set also carries a free-text failure `note`, which we keep, because those notes are ground truth the calculations cannot reproduce.

In [ ]:
OLD_DATA_URL = ("https://raw.githubusercontent.com/andrewvoss8-boop/"
                "core-me-data-science-activities-public/main/data/I_beam_data_2var.csv")

NEW_DATA_URL   = ("https://raw.githubusercontent.com/andrewvoss8-boop/"
                  "core-me-data-science-activities-public/main/data/purdue_beams")
NEW_DATA_LOCAL = "beams from zach harbin - beam_test_36301_summary.csv"   # local fallback (same file)

def _load(url, local, label):
    try:
        df = pd.read_csv(url); print(f"{label}: loaded from GitHub.")
    except Exception:
        df = pd.read_csv(local); print(f"{label}: GitHub link not set/reachable -> using local '{local}'.")
    return df

# ---- Original (16x25) ----
old = _load(OLD_DATA_URL, "../data/I_beam_data_2var.csv", "Original")
old = old.dropna(subset=["b_web_mm","H_web_mm","Strength N"])
old = old[(old["b_web_mm"] > 0) & (old["H_web_mm"] > 0)].copy()
df_old = pd.DataFrame({
    "box":   "Original (16x25, old stand)",
    "b":     old["b_web_mm"].values,
    "H_web": old["H_web_mm"].values,
    "P_meas":old["Strength N"].values,
    "note":  old.get("Column 1", pd.Series([""]*len(old))).fillna("").astype(str).values,
})

# ---- New Purdue (12x18) ----
new = _load(NEW_DATA_URL, NEW_DATA_LOCAL, "Purdue")
df_new = pd.DataFrame({
    "box":   "Purdue (12x18, Instron)",
    "b":     new["b"].values,
    "H_web": new["H"].values,
    "P_meas":new["strength_N"].values,
    "note":  "",
})

data = pd.concat([df_old, df_new], ignore_index=True)
print(f"\n{len(data)} beams total  ({len(df_old)} original + {len(df_new)} Purdue)")
data.head()


## 3. Section geometry

All three failure calculations need section properties. For a doubly-symmetric I-section with web $b\times H_\text{web}$ and two flanges $B\times t_f$:

- $I_x$ — strong-axis second moment (bending)
- $I_y$ — weak-axis second moment (flange-dominated; controls LTB)
- $J$ — St-Venant torsion constant, **Roark/Timoshenko rectangle form** (each rectangle gets a finite-aspect-ratio correction $1-0.63\,\tfrac{t}{a}+0.052\,(\tfrac{t}{a})^5$). This is *not* the thin-wall $\tfrac13\sum a t^3$ — see the note in section 4c.
- $C_w$ — warping constant, $C_w = I_y (H_\text{web}+t_f)^2/4$
- $Q$ — first moment of area about the neutral axis (for the $VQ/Ib$ shear distribution)

All returned in SI (metres) so forces come out in newtons.

In [ ]:
def section_props(b, H_web, B, TH):
    """I-section properties. Inputs in mm; outputs SI. Doubly symmetric."""
    t_f = (TH - H_web) / 2.0
    b_, h_, B_, tf_ = b/1e3, H_web/1e3, B/1e3, t_f/1e3
    c = (TH/1e3) / 2.0                                              # extreme-fiber distance
    Ix = (b_*h_**3)/12 + 2*((B_*tf_**3)/12 + B_*tf_*(h_/2 + tf_/2)**2)   # strong axis
    Iy = (h_*b_**3)/12 + 2*(tf_*B_**3)/12                                # weak axis
    # torsion constant J -- Roark/Timoshenko sum of rectangles WITH the finite-width correction
    # factor beta = 1 - 0.63(t/a) + 0.052(t/a)^5  (a = long side, t = short side of each rectangle)
    def _beta(t, a): return 1 - 0.63*(t/a) + 0.052*(t/a)**5
    J  = (1/3)*_beta(b_, h_)*h_*b_**3 + (2/3)*_beta(tf_, B_)*B_*tf_**3   # web + two flanges
    Cw = Iy*(h_ + tf_)**2/4                                              # warping
    Q  = B_*tf_*(h_/2 + tf_/2) + b_*(h_/2)*(h_/4)                        # 1st moment about NA
    return dict(t_f=t_f, c=c, Ix=Ix, Iy=Iy, J=J, Cw=Cw, Q=Q, b=b_, h=h_)

# sanity check on one beam
section_props(2.0, 12.0, 12.0, 18.0)


## 4. The three failure modes

### 4a. Bending (flexural yield)

Three-point bending puts the maximum moment at midspan, $M_\text{max} = PL/4$. The outer fiber reaches yield when $\sigma = M c / I_x = \sigma_y$:

$$P_\text{bend} = \frac{4\,\sigma_y\,I_x}{c\,L}$$

**Assumptions:** linear-elastic up to yield, plane sections remain plane (Euler–Bernoulli), stress is purely axial at the outer fiber. Good for slender beams; degrades for very deep/short beams.

In [ ]:
def P_bending(p, L):
    """3-pt bending flexural yield: P = 4 sigma_y Ix / (c L)."""
    return 4*YIELD_STR*p["Ix"] / (p["c"] * L/1e3)


### 4b. Shear, three ways

We compute shear three ways, because the choice changes the answer.

**(i) Average web shear.** Treat the web as the shear carrier. The average shear stress is $\tau_\text{avg}=V/(b\,H_\text{web})$ with $V=P/2$. Using von Mises shear yield $\tau_y=\sigma_y/\sqrt3$:

$$P_\text{shear} = 2\,(b\,H_\text{web})\,\frac{\sigma_y}{\sqrt3}$$

Simple and conservative. It scales with web area $b\,H_\text{web}$, so thin short webs come out shear-critical.

**(ii) Peak elastic shear $VQ/Ib$.** The elastic distribution peaks at the neutral axis: $\tau_\text{max}=VQ/(I_x b)$. For a typical I-beam it sits just above the average. For thick-flange, short-web sections the two pull apart, and this version gives a higher capacity (about 1.5 to 2x). Lean on it and you conclude shear never governs.

**(iii) von Mises bending–shear interaction.** Bending and shear act together. A point with axial stress $\sigma$ and shear $\tau$ yields when $\sigma_{vm}=\sqrt{\sigma^2+3\tau^2}=\sigma_y$. Combine the two pure failure loads:

$$\frac{1}{P_{vm}^2}=\frac{1}{P_\text{bend}^2}+\frac{1}{P_\text{shear}^2}$$

The combined capacity drops below either pure mode when both matter.

The interaction earns its place on the short-web beams. The Purdue $H_\text{web}=8$ beams measured below their pure-bending number, and the interaction predicts that drop. It falls down on brittle fracture. The original "web exploded / snapped / sheared off" failures happened well below the von Mises yield load. Von Mises assumes a ductile material yielding; a printed PLA web is brittle and weak along its layer lines, so it cracks instead. No $\sigma_y$ formula sees that. The von Mises number is a ductile lower bound, not a prediction of a brittle web burst.

In [ ]:
def P_shear_avg(p):
    """Average web shear: P = 2 (b*H_web) * sigma_y/sqrt(3)."""
    return 2*YIELD_STR*p["b"]*p["h"]/np.sqrt(3)

def P_shear_VQ(p):
    """Peak elastic shear tau_max = VQ/(Ix b): P = 2 sigma_y Ix b / (sqrt(3) Q)."""
    return 2*YIELD_STR*p["Ix"]*p["b"] / (np.sqrt(3)*p["Q"])

def P_vonmises(P_bend, P_shear):
    """Bending-shear interaction: 1/Pvm^2 = 1/Pbend^2 + 1/Pshear^2."""
    return 1.0/np.sqrt(1/P_bend**2 + 1/P_shear**2)


### 4c. Lateral-torsional buckling (LTB): the beam tips over

A tall, thin-flanged beam can roll and twist sideways before it yields. The elastic critical moment (doubly-symmetric I, central point load, load at the **top flange**) reads cleaner if we name the bracket $R$:

$$M_{cr}=C_1\,\frac{\pi^2 E I_y}{L_b^{2}}\,\Big[\sqrt{R}\;-\;C_2\,z_g\Big]
\qquad\text{where}\qquad
R=\frac{C_w}{I_y}+\frac{L_b^{2}\,G J}{\pi^2 E I_y}+\big(C_2\,z_g\big)^2$$

Then the buckling load is $P_\text{LTB}=4\,M_n/L$ with a **yield cap** $M_n=\min(M_y,\,M_{cr})$ (a beam can't carry more than its yield moment).

**Every input here is an assumption, and there are a lot of them:**

| symbol | meaning | where it comes from |
|---|---|---|
| $E,\,G$ | moduli | material, section 1 |
| $I_y,\,J,\,C_w$ | weak-axis / torsion / warping stiffness | geometry, section 3 |
| $L_b = k\,L$ | **unbraced length** | the **test stand** (see below) |
| $C_1$ | moment-gradient factor | load pattern (1.35 for central point load) |
| $C_2,\,z_g$ | **load-height** factor and height | *where the load touches the beam* |
| fork ends | twist prevented, warping free at supports | idealization of the supports |

$z_g=+TH/2$ means the load is applied at the **top (compression) flange**, which is **destabilizing** and *lowers* $M_{cr}$.

In [ ]:
def M_cr(p, L, k, z_g_sign=+1):
    """Elastic critical moment, top-flange (destabilizing) load. See markdown for the formula."""
    Lb  = k*L/1e3
    z_g = z_g_sign*p["c"]                       # +TH/2 : load at top flange
    pre  = C1_LTB*np.pi**2*E_MODULUS*p["Iy"]/Lb**2
    root = np.sqrt(p["Cw"]/p["Iy"]
                   + (Lb**2*G_MODULUS*p["J"])/(np.pi**2*E_MODULUS*p["Iy"])
                   + (C2_LTB*z_g)**2)
    return pre*(root - C2_LTB*z_g)

def P_LTB(p, L, k):
    """Yield-capped buckling load: M_n = min(M_y, M_cr); P = 4 M_n / L."""
    My = YIELD_STR*p["Ix"]/p["c"]
    Mn = min(My, M_cr(p, L, k))
    return 4*Mn/(L/1e3)

# ---- the "vanilla" textbook LTB (the early Desmos / hand-calc version) ----
def M_cr_vanilla(p, L):
    """Basic LTB: uniform-moment base x C1, load at the SHEAR CENTER, NO warping,
       NO load-height, full unbraced span (k=1). M_cr = C1 (pi/L) sqrt(E Iy G J)."""
    Lb = L/1e3
    return C1_LTB*np.pi/Lb*np.sqrt(E_MODULUS*p["Iy"]*G_MODULUS*p["J"])

def P_LTB_vanilla(p, L):
    """Vanilla LTB load (no yield cap, as in the basic calc): P = 4 M_cr_vanilla / L."""
    return 4*M_cr_vanilla(p, L)/(L/1e3)


### A note on the LTB formula, and a simpler version you may have seen

A basic LTB calculation, like the early hand and Desmos work on this project, is

$$M_{cr}^{\text{basic}}=C_1\,\frac{\pi}{L}\sqrt{E\,I_y\,G\,J}\,,\qquad P=\frac{4M_{cr}^{\text{basic}}}{L}.$$

The version here adds three things on top, and they push $M_{cr}$ in different directions:

| modeling choice | basic version | this notebook | effect on $M_{cr}$ |
|---|---|---|---|
| **Torsion constant $J$** | same Roark/Timoshenko rectangle form *(if you used the $1-0.63\,t/a+\dots$ factors)* | **Roark/Timoshenko** rectangle form | — |
| **Warping** $C_w$ | omitted | **included** ($C_w/I_y$ term) | **raises** $M_{cr}$ (matters for short beams) |
| **Load height** $z_g$ | omitted (load at shear center) | **top-flange** $z_g=+TH/2$ | **lowers** $M_{cr}$ (destabilizing) |
| **Unbraced length** | full span $L_b=L$ | $L_b=kL$ | **raises** $M_{cr}$ when $k<1$ |

**Thin-wall vs Timoshenko $J$.** The thin-wall torsion constant $J=\tfrac13\sum a\,t^3$ keeps only the leading term as thickness/length goes to zero. The Roark/Timoshenko form multiplies each rectangle by $1-0.63\,\tfrac{t}{a}+0.052\big(\tfrac{t}{a}\big)^5$, always below 1. For the thin-flanged tippers the two agree to about 5%, so the calibrated $k$ does not move. For stocky sections (thick flanges or webs) the thin-wall value runs 25 to 40% high. We use the Timoshenko form throughout; it matches the project's Desmos calc.

*(Material note: the Desmos calc used $E=2.75$ GPa and $\nu=0.325$; this notebook uses $E=2.5$ GPa and $\nu=0.35$ to match the other project notebooks. $E$ only enters LTB, and the calibrated $k$ was fit at $E=2.5$ GPa, so changing one means re-fitting the other.)*

## 5. LTB and the test stand: Purdue Instron vs. the original

LTB depends on how the stand holds and loads the beam, not on the beam alone. The two datasets came from different stands, so the LTB inputs differ:

| assumption | Purdue (Instron, photographed) | Original stand |
|---|---|---|
| **Unbraced length** $L_b=kL$ | compression flange laterally **unbraced**; calibrated $k\approx0.33$ (lumps support + load-point restraint and inelastic reserve) | **calibrated to the 7 recorded tippers**: $k\approx0.56$ (warping + top-flange load, ~5% error) — larger than Purdue, i.e. less restrained |
| **Load height** $z_g$ | knife-edge on the **top flange** → destabilizing ($z_g=+TH/2$) | depends on that stand's loading head — may differ |
| **End restraint** | beam rests on plain rollers — light twist restraint | different seating/restraint |
| **Observed result** | tall-web beams **tipped over** (LTB confirmed) | mixed bending / shear / LTB |

One thing to notice. The old stand was less stable, with less lateral restraint and a larger $k$ (0.56) than the Purdue stand (0.33). Calibrating to the recorded tippers nails their loads to about 5%. But applying that one $k$ across the whole box predicts more LTB-governed beams than actually tipped. Many original beams sit right on the bend/LTB boundary, so the predicted split swings hard with $k$: about 12 LTB at $k=0.5$, around 21 at $k=0.56$. The recorded 8 tippers match neither count, because the box's dominant real failure is brittle web fracture, and that sits in none of these formulas (section 7.5). One effective length can match the beams that buckled, or the overall mode mix, but not both on a dataset this messy.

For students: the $k=0.33$ that fits the Purdue tippers should not carry over to the original data. Re-using a fixture-calibrated constant across stands is how a "validated" model ends up wrong. $k$ is an assumption you set, not a constant you inherit.

## 6. Compute every failure force and tabulate

For each beam we report all modes. `P_gov` is the governing (minimum of the three pure modes) and `gov_mode` is which one. `P_vm` is the bending–shear interaction; `P_shear_VQ` is the peak-shear alternative (for comparison).

In [ ]:
def compute_row(r):
    cfg = BOXES[r["box"]]
    p   = section_props(r["b"], r["H_web"], cfg["B"], cfg["TH"])
    L, k = cfg["L"], cfg["k"]
    Pb   = P_bending(p, L)
    Psa  = P_shear_avg(p)
    Psvq = P_shear_VQ(p)
    Pl   = P_LTB(p, L, k)
    Plv  = P_LTB_vanilla(p, L)
    Pvm  = P_vonmises(Pb, Psa)
    modes = {"bend": Pb, "shear": Psa, "LTB": Pl}
    gov   = min(modes, key=modes.get)
    return pd.Series(dict(t_f=p["t_f"], P_bend=Pb, P_shear_avg=Psa, P_shear_VQ=Psvq,
                          P_vm=Pvm, P_LTB=Pl, P_LTB_van=Plv, P_full=min(Pvm, Pl),
                          P_gov=min(modes.values()), gov_mode=gov))

res = data.join(data.apply(compute_row, axis=1))

# ---- classify the RECORDED failure note (original set only) into an observed mode ----
def classify_observed(note, H_web):
    s = str(note).lower()
    if any(k in s for k in ["sideways","tipped","stand heavily bent","bent remains"]): return "LTB"
    if "flange" in s and H_web >= 19: return "LTB"                       # thin-flange tip at high web
    if any(k in s for k in ["yield","symmetric","symmetic"]):           return "bend"
    if any(k in s for k in ["web","shear","explod","snap","sheared","split","explosion"]): return "shear"
    if "flange" in s: return "flange"
    return ""
res["obs_mode"] = res.apply(lambda r: classify_observed(r["note"], r["H_web"]), axis=1)

show = res[["box","b","H_web","t_f","P_bend","P_shear_avg","P_shear_VQ","P_LTB",
            "P_vm","P_full","P_gov","gov_mode","P_meas","obs_mode","note"]].copy()
for c in ["P_bend","P_shear_avg","P_shear_VQ","P_LTB","P_vm","P_full","P_gov","P_meas"]:
    show[c] = show[c].round(0)
show["t_f"] = show["t_f"].round(1)
pd.set_option("display.max_rows", 120)
show


### 6.5 Vanilla textbook formulas vs. our versions, side by side

Two of the three equations have a textbook ("vanilla") version and a project version, and the gap changes which beams look safe. This is the comparison students make.

In [ ]:
comp = res[["box","b","H_web"]].copy()
comp["shear_vanilla_VQ/Ib"]   = res.P_shear_VQ.round(0)
comp["shear_OURS_avgweb+vM"]  = res.P_vm.round(0)
comp["LTB_vanilla_k1_noWarp"] = res.P_LTB_van.round(0)
comp["LTB_OURS_warp+topload+calK"] = res.P_LTB.round(0)
comp["measured"] = res.P_meas.round(0)
print("SHEAR: peak VQ/Ib runs ~1.5-2x ABOVE our avg-web von Mises  -> vanilla shear can hide shear entirely.")
print("LTB:   vanilla (k=1, no warping, centroid load) runs ~3x BELOW ours -> vanilla over-predicts tipping.\n")
comp.head(20)


**Shear.** The vanilla peak $VQ/Ib$ gives a higher capacity than our average-web von Mises number, so a model built on it concludes shear never governs. Our version (average web shear combined with bending through the von Mises interaction) matches the short-web beams that measured below pure bending.

**LTB.** The vanilla calc (uniform-moment base, no warping, load at the centroid, full unbraced span $k=1$) runs about 3x too low for these short beams, so it predicts tipping nearly everywhere. Our version adds warping stiffness, the top-flange load, and a fixture-calibrated effective length, and only then does LTB land on the beams that tipped. Section 9 shows the difference moves the optimal design.

## 7. Actual vs. predicted, colored by what drives the failure

We plot the measured failure force (y) against the predicted governing force $\min(P_\text{bend},P_\text{shear},P_\text{LTB})$ (x). The dashed line is actual = predicted.

The color carries the failure mode. Each point is an RGB blend of how much each mode drives, from the inverse-capacity share, so the governing mode gets the brightest channel:

- 🔴 Red = shear
- 🟢 Green = LTB
- 🔵 Blue = bending

A beam in the bend–shear interaction zone comes out purple. A clean tipper is green, a clean bender blue. Marker shape tells the two datasets apart.

In [ ]:
def mode_rgb(P_bend, P_shear, P_LTB):
    """RGB = how much each mode drives (inverse-capacity share). R=shear, G=LTB, B=bend."""
    inv   = np.array([1/P_shear, 1/P_LTB, 1/P_bend])   # channels: (R, G, B)
    share = inv/inv.sum()
    return tuple(share/share.max())                    # normalize -> vivid dominant color

res["rgb"] = res.apply(lambda r: mode_rgb(r.P_bend, r.P_shear_avg, r.P_LTB), axis=1)

fig, ax = plt.subplots(figsize=(7.6, 7.2))
for box, cfg in BOXES.items():
    sub = res[res.box == box]
    ax.scatter(sub.P_gov, sub.P_meas, c=list(sub.rgb), marker=cfg["marker"],
               s=95, edgecolor="k", linewidth=0.4, label=box)
hi = max(res.P_meas.max(), res.P_gov.max())*1.1
ax.plot([0, hi], [0, hi], "k--", lw=1, alpha=0.6, label="actual = predicted")
ax.set_xlim(0, hi); ax.set_ylim(0, hi)
ax.set_xlabel("Predicted failure force  =  min(P_bend, P_shear, P_LTB)   [N]")
ax.set_ylabel("Actual measured failure force   [N]")
ax.set_title("Actual vs predicted failure force\ncolor = driving mode  (R = shear, G = LTB, B = bending)")
ax.legend(loc="upper left", fontsize=9); ax.grid(alpha=0.3)

# simple RGB key
for txt, rgb, yy in [("shear",(1,0,0),0.93), ("LTB",(0,0.7,0),0.88), ("bending",(0,0,1),0.83)]:
    ax.scatter([], [], color=rgb, s=80, label=None)
    ax.text(0.965, yy, txt, color=rgb, transform=ax.transAxes, ha="right",
            fontsize=10, fontweight="bold")
plt.tight_layout(); plt.show()


### Reading the plot
- On the dashed line: the governing calc got the failure load right.
- Above the line: the beam held more than predicted, so the calc was conservative.
- Below the line: the beam was weaker than predicted, and the calc missed a mode. The original-set points down here are the brittle web-fracture beams the shear calc cannot see.

The next cell switches the x-axis to the full capacity, $\min(P_{vm},\,P_\text{LTB})$: the bending–shear interaction together with the LTB cap.

We need both terms. $P_{vm}$ combines bending and shear but has no LTB term, so on its own it over-predicts the tall-web tippers, reporting their high bend–shear capacity and ignoring the buckling that limited them. Taking $\min(\cdot,P_\text{LTB})$ puts those beams back on the line. The interaction pulls short-web beams down toward reality, and the LTB cap holds the tall-web beams.

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 7.2))
for box, cfg in BOXES.items():
    sub = res[res.box == box]
    ax.scatter(sub.P_full, sub.P_meas, c=list(sub.rgb), marker=cfg["marker"],
               s=95, edgecolor="k", linewidth=0.4, label=box)
hi = max(res.P_meas.max(), res.P_full.max())*1.1
ax.plot([0, hi], [0, hi], "k--", lw=1, alpha=0.6, label="actual = predicted")
ax.set_xlim(0, hi); ax.set_ylim(0, hi)
ax.set_xlabel("Predicted failure force  =  min(P_vm, P_LTB)   [N]")
ax.set_ylabel("Actual measured failure force   [N]")
ax.set_title("Complete model: bending-shear von Mises interaction AND the LTB cap\n"
             "min(P_vm, P_LTB) -- LTB-governed (green) beams now stay on the line")
ax.legend(loc="upper left", fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 7.5 The calculations vs. what was recorded (original set)

The original beams carry hand-written failure notes, the real ground truth. We sorted each note into an observed mode (`obs_mode`) and compare it to the predicted `gov_mode`. Where the two disagree, the calculation is missing physics.

In [ ]:
orig = res[res["box"].str.startswith("Original") & (res["obs_mode"] != "")].copy()
ct = pd.crosstab(orig["obs_mode"], orig["gov_mode"], margins=True)
print("Rows = OBSERVED (from notes), Columns = PREDICTED (min of the calcs):\n")
print(ct)
agree = (orig["obs_mode"] == orig["gov_mode"]).mean()
print(f"\nPredicted mode matches recorded mode for only {agree*100:.0f}% of the original beams.")
print("Look at the OBSERVED 'shear' row: those brittle web-fracture failures get predicted as")
print("bending or LTB, because no sigma_y-based formula contains a brittle interlayer-fracture mode.")


The recorded failures are shear-heavy (brittle web explosions), but the calculations call most of those beams bending or LTB. That mismatch is the edge of what closed-form mechanics can do for a brittle, layered, printed material. Treat the von Mises and LTB numbers as a prior. The recorded notes are the ground truth.

## 8. What the model gets right, and where it breaks

Gets right:
- Bending governs most stocky beams, and the flexural-yield number is close.
- LTB flags the tall-web, thin-flange beams on the Purdue stand (they tipped), once $k$ is calibrated to that stand.
- The von Mises interaction explains why short-web beams fail below pure bending.

Breaks:
1. Brittle web fracture is not in the model. The original "web exploded/snapped" failures land below any $\sigma_y$ shear-yield load. Von Mises assumes ductile yielding; printed PLA is brittle and layered. These points sit below the parity line and no closed form catches them.
2. The LTB constant $k$ does not transfer between stands. $k=0.33$ is a Purdue-Instron fit; the original stand is different and uncalibrated. Read its LTB column as illustrative.
3. The shear formula choice matters. Average-web shear makes thin short webs shear-critical; the $VQ/Ib$ peak reports 1.5 to 2x more capacity and can hide shear. We use average-web for the governing calc and show $VQ/Ib$ for contrast.
4. No data yet at the extremes (very short webs, say). Predictions there are extrapolation.

### Questions for students
1. Find an original-set beam where `obs_mode = "shear"` but `gov_mode` is bend or LTB. Read its `note`. Why can't any of these three calcs predict that failure?
2. Sweep `K_ORIGINAL` over 0.33, 0.5, 0.7, 1.0 and recompute. How does the original box's LTB count change? What does that say about trusting a fixture-fit constant on a different stand?
3. Swap `P_shear_avg` for `P_shear_VQ` in `compute_row`. Does shear still drive any beam? Which shear idea do you trust, and for which shapes?
4. For a beam where bending and shear are within 5% of each other, is $P_{vm}$ above or below the measured value? What does the interaction add?

## 8.5 Student activity: compare the data to the theory, then tweak the constants

The constants are not handed down. You fit them to what you saw. The clearest case is the LTB effective length $k$. Change `k_try` below and watch the predicted-vs-measured fit on the tipped beams move. Students pick the $k$ that best reproduces the beams that tipped, then justify it against the fixture photo.

In [ ]:
def ltb_fit_report(k_try, box="Purdue (12x18, Instron)"):
    cfg = BOXES[box]; sub = res[res.box == box]
    tall = sub[sub.H_web >= sub.H_web.max() - 0.01]          # tall-web tippers, where LTB is the mode
    errs = []
    for _, r in tall.iterrows():
        p = section_props(r.b, r.H_web, cfg["B"], cfg["TH"])
        errs.append(abs(P_LTB(p, cfg["L"], k_try)/r.P_meas - 1))
    print(f"k_try = {k_try:.2f}:  mean |P_LTB / measured - 1| on the tall-web tippers = {np.mean(errs)*100:.0f}%")

print("Purdue stand — sweep the effective length and find the best fit to the beams that tipped:\n")
for k_try in [0.20, 0.33, 0.50, 1.00]:
    ltb_fit_report(k_try)
print("\nThe minimum-error k is the calibrated value. Students argue it against the fixture (top-flange,")
print("unbraced) and note that it does not transfer to the other stand.")


## 9. Pick a mass-optimized design from the theory

With a failure model in hand, students ask it for the best strength-to-weight design, then decide whether to trust it. We optimize over the original 16×25 space twice, once with our full LTB model and once with the vanilla LTB calc, and compare to the best beam actually built.

In [ ]:
KMASS = 0.2718   # g/mm^2, calibrated mass-per-area from the weighed original prints

def str_to_weight(b, Hw, box, vanilla_ltb=False):
    cfg = BOXES[box]; p = section_props(b, Hw, cfg["B"], cfg["TH"]); L = cfg["L"]
    Pvm = P_vonmises(P_bending(p, L), P_shear_avg(p))
    Pl  = P_LTB_vanilla(p, L) if vanilla_ltb else P_LTB(p, L, cfg["k"])
    strength = min(Pvm, Pl)
    mass = KMASS*(b*Hw + cfg["B"]*(cfg["TH"] - Hw))          # g
    return strength/mass

def optimize(box, vanilla_ltb=False):
    best = (None, None, -1)
    for b in np.arange(1.0, 8.01, 0.05):
        for Hw in np.arange(12.0, 23.01, 0.05):
            sw = str_to_weight(b, Hw, box, vanilla_ltb)
            if sw > best[2]: best = (b, Hw, sw)
    return best

box = "Original (16x25, old stand)"
b1, H1, sw1 = optimize(box, vanilla_ltb=False)
b2, H2, sw2 = optimize(box, vanilla_ltb=True)
print(f"OUR full model    -> optimum b={b1:.2f}, H_web={H1:.2f}  ->  str/w {sw1:.1f} N/g  (bending-governed)")
print(f"VANILLA LTB calc  -> optimum b={b2:.2f}, H_web={H2:.2f}  ->  str/w {sw2:.1f} N/g  (LTB-governed)")
print(f"Best ACTUALLY built (from data)  -> b=4.50, H_web=14.50  ->  str/w ~32.3 N/g")
print("\nThe vanilla LTB optimum flees to a heavy, thick-flange corner and predicts a str/w ceiling")
print("below a beam that was already measured.")


Our model's optimum, a lighter and taller-web design, sits a few percent above the best beam actually built. That is sensible headroom. The vanilla LTB calc sends the optimum to a heavy, thick-flanged corner and claims a strength-to-weight ceiling below a beam already in the dataset. The modeling choices back in section 4c change the recommended design.

Neither optimum sees the brittle web fracture that, in practice, punishes the thin-web designs the model likes. The tested beam nearest our predicted optimum exploded and came in about 10% low. The equations have taken us about as far as they can, which is where the GP comes in.

## 10. From equations to a Gaussian Process

We have a physics model that is useful but incomplete. It misses brittle fracture, its LTB term depends on the stand, the failure modes blur, and we have only a few dozen expensive tests. A Gaussian Process surrogate fits this case: it learns the strength surface from sparse data and reports its own uncertainty where the data is thin.

The companion notebook `ME323_Module1_Virtual_Lab_A` carries the GP and Bayesian-optimization mechanics. The arc continues there:

1. **Fit a plain GP** to the tested beams and read off a recommended design, with an uncertainty band rather than a single number. (This is ME 239 directly: a GP is conditioning a multivariate Gaussian on data.)
2. **Add physics.** Feed the model the mechanics it does know: the failure-mode capacities from this notebook ($P_\text{bend}$, $P_{vm}$, $P_\text{LTB}$) become features. Compare the plain GP to the physics-informed one. Where do the physics features sharpen the prediction, and where (the brittle-fracture corner) do they mislead?
3. **Explore vs. exploit.** The model's uncertainty is as useful as its mean. Test the design that looks best now, or the one you know least about?
4. **MUI vs. EI.** Maximum Upper Interval ($\mu+\psi\sigma$) against Expected Improvement. They can point at different next beams, and the choice sets how much risk you take for how much potential gain.
5. **Write the memo.** Model choice, noise assumption, acquisition rule, the recommendation, and its limits. That is the graded piece, and where the judgment lives.

The physics narrows the space and gives the GP good features. The GP carries the uncertainty the physics can't, folds in the data, and turns "which beam next?" into a defensible call under risk. That loop is the module.

## 11. References (equation-by-equation)

Each formula maps to a primary source below. We give chapter, clause, or table names rather than page numbers, since pagination varies by edition. Please confirm the exact page, equation, or table and the journal volume and page numbers against the editions you hold.

**Section 3 — Section properties**
- $I_x,\,I_y$ (composite area, parallel-axis theorem) — Gere & Goodno, *Mechanics of Materials* (Cengage), "Moments of Inertia of Plane Areas"; Beer, Johnston et al., *Vector Mechanics for Engineers: Statics*, "Moments of Inertia."
- Torsion constant $J=\tfrac13\big(1-0.63\tfrac{t}{a}+0.052(\tfrac{t}{a})^5\big)\,a\,t^3$ per rectangle — Young, Budynas & Sadegh, *Roark's Formulas for Stress and Strain* (8th ed.), "Torsion," rectangular-section table (the 0.63 / 0.052 coefficients are Roark's, algebraically identical to $ab^3[\tfrac13-0.21\tfrac{b}{a}(1-\tfrac{b^4}{12a^4})]$); underlying elasticity solution: Timoshenko & Goodier, *Theory of Elasticity* (3rd ed.), "Torsion of Prismatical Bars." Open-section sum $J=\sum J_i$ — Megson, *Aircraft Structures for Engineering Students*, "Torsion of open-section beams."
- Warping constant $C_w=I_y h_0^2/4$ (doubly-symmetric I, $h_0$ = flange-centroid spacing) — Salmon, Johnson & Malhas, *Steel Structures: Design and Behavior*; ANSI/AISC 360 member-property definitions; Timoshenko & Gere, *Theory of Elastic Stability* (non-uniform/warping torsion).

**Section 4a — Bending (flexural yield)**
- Flexure formula $\sigma=Mc/I$ — Gere & Goodno / Hibbeler, *Mechanics of Materials*, "Bending Stresses in Beams."
- Simply-supported central point load $M_{\max}=PL/4,\ V=P/2$ — standard beam tables (AISC *Steel Construction Manual*, "Beam Diagrams and Formulas"; Roark's, simply-supported-beam table).
- 3-point flexural test of polymers (method/span/stress) — ASTM D790, *Standard Test Methods for Flexural Properties of Unreinforced and Reinforced Plastics and Electrical Insulating Materials.*

**Section 4b — Shear**
- Transverse shear $\tau=VQ/(Ib)$ (Zhuravskii/Jourawski) — Gere & Goodno / Hibbeler, "Shear Stresses in Beams."
- Average web shear $V/A_w$ — ANSI/AISC 360, Chapter G, "Design of Members for Shear."
- von Mises criterion $\sigma_{vm}=\sqrt{\sigma^2+3\tau^2}$, shear yield $\tau_y=\sigma_y/\sqrt3$ — R. von Mises (1913), *Nachr. Ges. Wiss. Göttingen, Math.-Phys. Kl.*, 582–592; textbook treatment: Dowling, *Mechanical Behavior of Materials*, "Yielding Criteria"; Boresi & Schmidt, *Advanced Mechanics of Materials.*
- Bending–shear interaction $1/P_{vm}^2=1/P_\text{bend}^2+1/P_\text{shear}^2$ — *derived* by applying von Mises to combined $\sigma,\tau$ at a point, $(\sigma/\sigma_y)^2+(\tau/\tau_y)^2=1$ (Boresi & Schmidt, combined stresses). **Note for reviewer:** this is a derived elliptical interaction, not a tabulated code equation.

**Sections 4c & 5 — Lateral-torsional buckling**
- Fundamental elastic $M_{cr}=\tfrac{\pi}{L}\sqrt{EI_yGJ}\,\sqrt{1+\pi^2EC_w/(L^2GJ)}$ — Timoshenko & Gere, *Theory of Elastic Stability* (2nd ed.), Ch. 6, "Lateral Buckling of Beams."
- 3-factor form with $C_1,C_2,z_g$ and the values $C_1\!\approx\!1.35,\ C_2\!\approx\!0.55$ for a central point load — EN 1993-1-1 (Eurocode 3) with NCCI **SN003a-EN-EU**, *Elastic critical moment for lateral torsional buckling* (Access Steel); Galambos (ed.), *Guide to Stability Design Criteria for Metal Structures* (SSRC), 6th ed.
- Moment-gradient factor $C_b/C_1$ — ANSI/AISC 360, Chapter F (Eq. F1-1); Salmon & Johnson.
- Load-height effect (top-flange load destabilizing; position coefficients) — Timoshenko & Gere, Ch. 6 (load at centroid vs. flanges); Trahair, *Flexural–Torsional Buckling of Structures.*
- Unbraced length $L_b=kL$ — ANSI/AISC 360, Ch. F; SSRC Guide. **Here $k$ is fixture-calibrated to test data, not taken from a table** (sections 5, 7.5).

**Material assumptions (printed PLA)**
- $E\approx2.5$–$3.5$ GPa, $\sigma_y\approx50$–$70$ MPa — filament manufacturer TDS; Farah, Anderson & Langer (2016), "Physical and mechanical properties of PLA, and their functions in widespread applications," *Advanced Drug Delivery Reviews* 107, 367–392.
- FDM interlayer anisotropy / brittle Z-direction fracture (the unmodeled "web-explosion" mode) — Ahn, Montero, Odell, Roundy & Wright (2002), "Anisotropic material properties of fused deposition modeling ABS," *Rapid Prototyping Journal* 8(4), 248–257; Chacón, Caminero, García-Plaza & Núñez (2017), "Additive manufacturing of PLA structures using FDM: effect of process parameters on mechanical properties and their optimal selection," *Materials & Design* 124, 143–157.